# Milestone 2 lab, notebook 7 of 8: Counted twice

## Where we are

| Notebook | What you do |
|---|---|
| 1. Ask only for what changed | Read the orders that changed, not all of them |
| 2. Remember where you stopped | Keep a watermark. See what happens when it moves too early |
| 3. When the API says no | Meet the rate limit and a server error. Wait, and try again |
| 4. Land it raw | Put every order in Postgres, exactly as it arrived |
| 5. Pull or push | A short lab: let the order system send the changes to you |
| 6. What you built has a name | Change data capture, and what this kind cannot see |
| **7. Counted twice** (you are here) | Run a few mornings. Watch the report go wrong |
| 8. The production version | The same logic as a tested program |

The load is small, it survives a bad API, and it keeps everything. Time to read the manager's report from it.

This notebook uses the finished pipeline from the `daily_orders/` folder: the steps of notebooks 1 to 4, put together.

In [1]:
import pandas as pd

from lab_helpers import LAB_DATABASE, api, bars, lab_warehouse, money, set_morning, show_doc, table, true_day_totals

from daily_orders import pipeline, warehouse          # the production code of this milestone
from daily_orders.report import daily_report, day_totals

lab_warehouse(fresh=True).close()            # start from an empty lab database
con = warehouse.connect(LAB_DATABASE)        # the production code makes its tables

## Step 1: Monday, then Tuesday

In [2]:
for day in ["2026-06-08", "2026-06-09"]:
    set_morning(day)
    result = pipeline.run(api, con)
    print(f"   run {result.run_id}: landed {result.rows_landed} rows in {result.api_calls} calls. Watermark -> {result.watermark_after}\n")

It is now Monday 2026-06-08 07:00:00 in Arden.
   run 2026-06-08_1: landed 226 rows in 10 calls. Watermark -> 2026-06-08T03:48:53Z

It is now Tuesday 2026-06-09 07:00:00 in Arden.
   run 2026-06-09_1: landed 29 rows in 2 calls. Watermark -> 2026-06-09T04:17:31Z



226 rows, then 29. Exactly what you expect.

## Step 2: The report, beside the truth

The report is a view on the landing table. For the truth, do what milestone 1 did: read **every** order now and add each day up by hand. `true_day_totals` does that.

In [3]:
def compare():
    """The report from the warehouse, beside the truth from a full read of the source."""
    report, truth = day_totals(daily_report(con)), true_day_totals()
    rows = []
    for day, right in truth.items():
        shown = report.get(day, {"orders": 0, "revenue_cents": 0})
        wrong = (shown["orders"], shown["revenue_cents"]) != (right["orders"], right["revenue_cents"])
        rows.append({
            "day": day,
            "report": f"{shown['orders']} orders, {money(shown['revenue_cents'])}",
            "truth": f"{right['orders']} orders, {money(right['revenue_cents'])}",
            "too_much": money(shown["revenue_cents"] - right["revenue_cents"]) if wrong else "",
            "": "❌" if wrong else "✅",
        })
    return pd.DataFrame(rows)

set_morning("2026-06-09")
compare()

It is now Tuesday 2026-06-09 07:00:00 in Arden.


,day,report,truth,too_much,
0,2026-06-01,"28 orders, $730.76","28 orders, $730.76",,✅
1,2026-06-02,"32 orders, $751.89","32 orders, $751.89",,✅
2,2026-06-03,"35 orders, $832.47","35 orders, $832.47",,✅
3,2026-06-04,"30 orders, $591.98","30 orders, $591.98",,✅
4,2026-06-05,"38 orders, $985.79","37 orders, $971.04",$14.75,❌
5,2026-06-06,"33 orders, $786.71","33 orders, $786.71",,✅
6,2026-06-07,"33 orders, $941.47","30 orders, $807.11",$134.36,❌
7,2026-06-08,"25 orders, $591.75","25 orders, $591.75",,✅


Two days are wrong, and both are **too high**. Friday shows 38 orders. Arden took 37.

No error was shown. The pipeline ran perfectly.

## Step 3: Find the extra orders

If a day has too many orders, some order must be there more than once. Ask the warehouse.

In [4]:
table(con, """
    SELECT order_id, order_date, count(*) AS times_in_table
    FROM analytics.orders_as_landed
    GROUP BY order_id, order_date
    HAVING count(*) > 1
    ORDER BY order_id
""")

,order_id,order_date,times_in_table
0,1130,2026-06-05,2
1,1224,2026-06-07,2
2,1225,2026-06-07,2
3,1226,2026-06-07,2


Four orders are in the table twice. Look at two of them.

In [5]:
table(con, """
    SELECT landing_id, order_id, order_date, status, total_cents, refunded_cents, run_id
    FROM analytics.orders_as_landed
    WHERE order_id IN (1130, 1224)
    ORDER BY order_id, landing_id
""")

,landing_id,order_id,order_date,status,total_cents,refunded_cents,run_id
0,137,1130,2026-06-05,delivered,1475,0,2026-06-08_1
1,232,1130,2026-06-05,delivered,1475,578,2026-06-09_1
2,224,1224,2026-06-07,placed,7018,0,2026-06-08_1
3,229,1224,2026-06-07,delivered,7018,0,2026-06-09_1


- **Order 1130** was landed on Monday, before its refund. It changed, so Tuesday's run landed it again. The report adds $14.75 **and** $8.97.
- **Order 1224** was placed late on Sunday. Monday's run saw it as `placed`. It was delivered on Monday, so Tuesday's run landed it again.

Each row is true. Notebook 4 was proud of this: the old version is kept beside the new one. The mistake is in the report.

In [6]:
show_doc("LLD.md", "4. The warehouse (`warehouse.py`)")

**From `docs/LLD.md`**

> ## 4. The warehouse (`warehouse.py`)
> 
> One Postgres database, `arden_m2`, with three schemas.
> 
> *(There is a diagram here. Open `docs/LLD.md` to see it.)*
> 
> | Schema | Holds | Who writes |
> |---|---|---|
> | `raw` | Data exactly as it arrived | Only the load. Only by adding rows |
> | `ops` | What the pipeline remembers about itself | Only the load |
> | `analytics` | What people read | Nobody. It holds views |
> 
> Two views sit on the landing table:
> 
> | View | Grain | What it does |
> |---|---|---|
> | `analytics.orders_as_landed` | One row per **landed row** | Turns the JSON into columns. Works out `order_date` in Arden's time zone |
> | `analytics.daily_orders` | One row per day, per city | The report. It adds up every landed row |
> 
> The grain of the first view is the trap. It is not one row per order.

Read the last line of that section again. The view has one row per **landed row**. The report adds it up as if it had one row per **order**. That is the grain mistake from milestone 1, in a new place.

## Step 4: Run it twice

In milestone 1, running the same morning twice changed nothing. Try it here. It is still Tuesday.

In [7]:
set_morning("2026-06-09")
result = pipeline.run(api, con)
print(f"run {result.run_id}: landed {result.rows_landed} row")

table(con, """
    SELECT landing_id, order_id, order_date, status, total_cents, run_id
    FROM analytics.orders_as_landed
    WHERE order_id = (SELECT order_id FROM analytics.orders_as_landed WHERE run_id = '2026-06-09_2')
    ORDER BY landing_id
""")

It is now Tuesday 2026-06-09 07:00:00 in Arden.
run 2026-06-09_2: landed 1 row


,landing_id,order_id,order_date,status,total_cents,run_id
0,255,1251,2026-06-08,placed,3765,2026-06-09_1
1,256,1251,2026-06-08,placed,3765,2026-06-09_2


One more row. It is the boundary order from notebook 2: its `updated_at` equals the watermark, so it is read again, and append adds it again.

In [8]:
compare()

,day,report,truth,too_much,
0,2026-06-01,"28 orders, $730.76","28 orders, $730.76",,✅
1,2026-06-02,"32 orders, $751.89","32 orders, $751.89",,✅
2,2026-06-03,"35 orders, $832.47","35 orders, $832.47",,✅
3,2026-06-04,"30 orders, $591.98","30 orders, $591.98",,✅
4,2026-06-05,"38 orders, $985.79","37 orders, $971.04",$14.75,❌
5,2026-06-06,"33 orders, $786.71","33 orders, $786.71",,✅
6,2026-06-07,"33 orders, $941.47","30 orders, $807.11",$134.36,❌
7,2026-06-08,"26 orders, $629.40","25 orders, $591.75",$37.65,❌


Now Monday is wrong too. **Every extra run makes the report worse.** Someone who clicks "run" twice changes what the manager sees.

## Step 5: Two more mornings

Run Wednesday and Thursday.

In [9]:
for day in ["2026-06-10", "2026-06-11"]:
    set_morning(day)
    result = pipeline.run(api, con)
    print(f"   run {result.run_id}: landed {result.rows_landed} rows in {result.api_calls} calls\n")

It is now Wednesday 2026-06-10 07:00:00 in Arden.
   run 2026-06-10_1: landed 38 rows in 2 calls

It is now Thursday 2026-06-11 07:00:00 in Arden.
   run 2026-06-11_1: landed 49 rows in 3 calls



Thursday landed more rows than any incremental run so far. Look at its pages.

In [10]:
table(con, """
    SELECT l.page_no, count(*) AS rows_landed, count(DISTINCT (l.payload->>'order_id')) AS different_orders,
           min((l.payload->>'order_id')::int) AS first_order, max((l.payload->>'order_id')::int) AS last_order
    FROM raw.orders_landing AS l
    WHERE l.run_id = '2026-06-11_1'
    GROUP BY l.page_no ORDER BY l.page_no
""")

,page_no,rows_landed,different_orders,first_order,last_order
0,1,25,25,1281,1310
1,2,12,12,1304,1317
2,3,12,12,1304,1317


Pages 2 and 3 hold the same orders. On this morning **the order system sent one page twice**. That happens with real APIs: a retry, a timeout, a bug on their side. Your load did nothing wrong, and append stored it all.

`page_no` is load metadata. Without it you could not have seen this.

## Step 6: How wrong is it now?

In [11]:
set_morning("2026-06-11")
result = compare()
result[result[""] == "❌"]

It is now Thursday 2026-06-11 07:00:00 in Arden.


,day,report,truth,too_much,
4,2026-06-05,"38 orders, $985.79","37 orders, $971.04",$14.75,❌
6,2026-06-07,"33 orders, $941.47","30 orders, $807.11",$134.36,❌
7,2026-06-08,"33 orders, $808.62","24 orders, $573.49",$235.13,❌
8,2026-06-09,"33 orders, $736.74","31 orders, $714.80",$21.94,❌
9,2026-06-10,"47 orders, $1,220.41","35 orders, $875.89",$344.52,❌


In [12]:
counts = table(con, "SELECT count(*) AS rows, count(DISTINCT order_id) AS orders FROM analytics.orders_as_landed")
bars({"Rows in the landing table": int(counts["rows"][0]), "Different orders": int(counts["orders"][0])})

Rows in the landing table  ████████████████████████████████████████ 343
Different orders           █████████████████████████████████████ 317


## Why this happens: append

Everything this milestone does is safe from **loss**:

| Choice | Why | Its side effect |
|---|---|---|
| The filter includes the boundary | So a row is never skipped | The boundary row is read again |
| A failed call is retried | So a hiccup does not stop the report | The same page can arrive twice |
| Changed orders are loaded again | So a refund is not missed | The order arrives a second time |

Each choice trades a possible loss for a possible **repeat**. That is the right trade. But the write is an **append**, and append cannot tell that it has seen a row before. So every repeat becomes a second row, and the report counts it.

The cure is not to stop the repeats. It is to make the write not care about them.

> **Still wrong:** a changed order is now in the table twice, and the report counts it twice. Run the load two times and it is worse.

Milestone 3 fixes it: one order, one row, however often it arrives.

In [13]:
con.close()

## What you now know

- An order that changes is landed again. Append keeps both versions, and the report adds both.
- Running a morning twice adds the boundary order again. A repeated page doubles a whole page.
- The landing table is right. The grain of the report is wrong.
- A load that is safe from loss must expect repeats.

**Say it in an interview:** "My incremental load was at-least-once on purpose: an inclusive watermark, retries, and reloading changed rows. With an append-only write, each repeat became a duplicate row, and the daily report double counted. I kept the raw table as it was and fixed the layer above it."

Next: [notebook 8](08_the_production_version.ipynb) shows this milestone as a tested program.